Train/Test Split

In [76]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

#insira aqui o caminho para os dados limpos que você quer modelar:
DADOS_LIMPOS = Path("C:/Users/Gamer GTX/sintese/projeto trainee/Projeto-Trainee-I-Dados-2026.1/dados/dados_limpos.csv")

if not DADOS_LIMPOS.exists():
    raise FileNotFoundError("dados/dados_limpos.csv")

transpondo a coluna "Class" no indice 0 do dataframe:

In [77]:
df_limpo = pd.read_csv(DADOS_LIMPOS)
transpondo_class = df_limpo.pop("Class")
df_limpo.insert(0, "Class", transpondo_class)


C:\Users\Gamer GTX\AppData\Local\Temp\ipykernel_10916\36720065.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_limpo.insert(0, "Class", transpondo_class)


Em X, temos os genes como variaveis preditoras. Em y, temos a variavel target (tipo de tumor)

Estabelecendo em 20% o volume de dados para teste e 80% para treino

In [78]:
genes = df_limpo.drop(columns=["Class"])
tumores = df_limpo["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    genes,
    tumores,
    test_size=0.2,
    random_state=42
)

Normalizando os dados

In [79]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
scaler.set_output(transform="pandas")

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Para realização do Label enconding verificamos inicialmente se existe alguma variável categórica no conjunto de treinamento X, e depois aplicamos o label enconding para o conjunto de dados Y, que representam o target: tipos de cancêr

In [80]:
colunas_categoricas = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Quantidade de colunas categóricas em X: {len(colunas_categoricas)}")

Quantidade de colunas categóricas em X: 0


In [81]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

for idx, classe in enumerate(le.classes_):
    print(f"{classe} -> {idx}")

BRCA -> 0
COAD -> 1
KIRC -> 2
LUAD -> 3
PRAD -> 4


In [82]:
import numpy as np
import shap 
from sklearn.ensemble import RandomForestClassifier

rf_inicial = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
rf_inicial.fit(X_train_scaled, y_train)

importancias = pd.Series(rf_inicial.feature_importances_, index=X_train_scaled.columns)

features_grande = importancias[importancias > 0.001].index.to_list()

X_train_scaled = X_train_scaled[features_grande]
X_test_scaled = X_test_scaled[features_grande]

print(X_train_scaled.shape)

(640, 324)


Redução do dataset através de RFC utilizando valores de importância acima de 0.5%, para evitar o descarte de variáveis que possam ser relevantes para o modelo e retirar apenas os ruídos.

In [83]:
rf_final = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
rf_final.fit(X_train_scaled, y_train)

amostra_shap = X_train_scaled.sample(n=min(5000, len(X_train_scaled)), random_state=42)

explainer = shap.TreeExplainer(rf_final)
shap_valores = explainer(amostra_shap)
vals = shap_valores.values
if vals.ndim == 3:
    media_shap = np.abs(vals).mean(axis=(0,2))
importancia_shap = pd.Series(media_shap, index=X_train_scaled.columns)
features_finais = importancia_shap.sort_values(ascending=False).head(50).index.to_list()

X_train_scaled = X_train_scaled[features_finais]
X_test_scaled = X_test_scaled[features_finais]



Com o novo X_train utiliza-se RFC novamente para prever as variáveis úteis, depois foi utilizado o SHAP para obter as maiores importâncias das features restantes e assim reduzir o X_train_scaled para as 50 features mais relevantes

In [84]:
for i, gene in enumerate(features_finais,1):
    print(f"{i} : {gene}")

1 : gene_18746
2 : gene_17801
3 : gene_6748
4 : gene_7964
5 : gene_15591
6 : gene_3737
7 : gene_9176
8 : gene_6611
9 : gene_9652
10 : gene_219
11 : gene_15898
12 : gene_12983
13 : gene_6530
14 : gene_14092
15 : gene_5729
16 : gene_220
17 : gene_5407
18 : gene_11910
19 : gene_17905
20 : gene_15896
21 : gene_8014
22 : gene_17664
23 : gene_18178
24 : gene_15899
25 : gene_10731
26 : gene_5578
27 : gene_16358
28 : gene_12078
29 : gene_3453
30 : gene_16132
31 : gene_4833
32 : gene_6816
33 : gene_2288
34 : gene_15893
35 : gene_3439
36 : gene_18381
37 : gene_7965
38 : gene_8801
39 : gene_6733
40 : gene_7238
41 : gene_9184
42 : gene_6355
43 : gene_16372
44 : gene_2037
45 : gene_15897
46 : gene_11349
47 : gene_16342
48 : gene_1122
49 : gene_11550
50 : gene_2747


Treinamento do algoritmo

In [ ]:
modelo_final = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)

# Treinando o modelo com as 50 features selecionadas
modelo_final.fit(X_train_scaled, y_train)

#Realizando as predições com os dados de teste iniciais
y_pred = modelo_final.predict(X_teste_scaled)